# Vertex AI 計算パイプラインの実行

このNotebookは、2つの数値を乗算し、加算し、その結果の差を計算するVertex AIパイプラインをセットアップして実行する手順を示します。

## 1. 依存関係のインストール

In [ ]:
!pip install --quiet --upgrade google-cloud-aiplatform kfp

## 2. パイプラインの構成

以下のセルで、ご自身のGoogle CloudプロジェクトID、リージョン、およびGoogle Cloud Storage（GCS）バケットを設定してください。

In [ ]:
# --- ユーザー構成 --- 
PROJECT_ID = "your-gcp-project-id"      # <-- 置き換えてください
REGION = "your-gcp-region"          # <-- 置き換えてください (例: "us-central1")
PIPELINE_ROOT = "gs://your-gcs-bucket/pipeline-root" # <-- 置き換えてください
# --------------------------

# Google Cloud認証 (必要な場合)
import sys
if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

## 3. パイプラインのコンパイル

パイプラインを定義し、JSONファイルにコンパイルします。

In [ ]:
from kfp import compiler
from pipeline import calculation_pipeline

PIPELINE_JSON = "calculation_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=calculation_pipeline,
    package_path=PIPELINE_JSON
)

print(f"Pipeline compiled to {PIPELINE_JSON}")

## 4. パイプラインジョブの実行

コンパイルされたパイプラインをVertex AIに送信して実行します。

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)

# パイプラインジョブを作成
job = aiplatform.PipelineJob(
    display_name="calculation-pipeline-notebook-run",
    template_path=PIPELINE_JSON,
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        'num1': 20,
        'num2': 4
    }
)

# パイプラインジョブを送信
job.submit()

print(f"Pipeline job submitted. View it in the console: {job.dashboard_uri}")